# EDA 05 — Ingredient Mapping & Candidate Generation

This notebook is the cleaned final version of the previous 27-cell mapping workflow.

### Pipeline
**RecipeDB → Carbon Footprint → FlavorDB → molecular similarity → candidate generation → recipe co-occurrence → sustainability screening**

The exploratory cells that only investigated substring noise, repeated similarity variants, and intermediate diagnostics have been condensed. Their useful conclusions are retained in the final mapping/similarity outputs.

> **Important:** molecular similarity is treated as a candidate-generation signal, not as a validated measure of culinary substitutability.


In [33]:
# ================================================================
# CELL 1 — SETUP & LOAD DATA
# ================================================================

from pathlib import Path
from itertools import combinations
from collections import Counter
import math
import re

import numpy as np
import pandas as pd

BASE = Path.cwd().parent
if not (BASE / "dataset").exists():
    BASE = BASE.parent

DATASET = BASE / "dataset"
RECIPE_DB = DATASET / "RecipeDB_full"
PROCESSED = DATASET / "processed"
RESULTS = BASE / "EDA" / "results" / "mapping"

RESULTS.mkdir(parents=True, exist_ok=True)

recipe_ingredients = pd.read_csv(
    RECIPE_DB / "recipe_ingredients.csv"
)

carbon = pd.read_csv(
    PROCESSED / "SustainableFoodDB_CarbonFootprints.csv"
)

flavor_entities = pd.read_csv(
    PROCESSED / "FlavorDB_entities.csv"
)

flavor_molecules = pd.read_csv(
    PROCESSED / "FlavorDB_molecules.csv"
)

flavor_links = pd.read_csv(
    PROCESSED / "FlavorDB_entity_molecules.csv"
)

recipe_ingredients["ingredient_name"] = (
    recipe_ingredients["ingredient_name"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

recipe_ingredients = recipe_ingredients[
    recipe_ingredients["ingredient_name"] != ""
].copy()

print("RecipeDB ingredients :", recipe_ingredients.shape)
print("Carbon dataset       :", carbon.shape)
print("FlavorDB entities    :", flavor_entities.shape)
print("FlavorDB molecules   :", flavor_molecules.shape)
print("FlavorDB links       :", flavor_links.shape)

print("\nUnique RecipeDB ingredient types:",
      recipe_ingredients["ingredient_name"].nunique())
print("RecipeDB ingredient occurrences:",
      len(recipe_ingredients))


RecipeDB ingredients : (161934, 10)
Carbon dataset       : (7753, 5)
FlavorDB entities    : (934, 6)
FlavorDB molecules   : (1788, 23)
FlavorDB links       : (60075, 2)

Unique RecipeDB ingredient types: 5693
RecipeDB ingredient occurrences: 161934


## 2. Exact mapping baseline

First establish what can be matched without semantic normalization. This gives us the baseline against which controlled normalization is measured.


In [34]:
# ================================================================
# CELL 2 — EXACT MAPPING BASELINE
# ================================================================

recipe_names = sorted(
    recipe_ingredients["ingredient_name"].dropna().unique()
)

def clean_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

recipe_names = [clean_name(x) for x in recipe_names]

# ---------- Carbon ----------
carbon_map = (
    carbon.assign(
        ingredient_key=carbon["Ingredient"].map(clean_name)
    )
    .dropna(subset=["ingredient_key"])
)

carbon_lookup = {}
for _, row in carbon_map.iterrows():
    key = row["ingredient_key"]
    if key and key not in carbon_lookup:
        carbon_lookup[key] = row["Ingredient"]

carbon_exact = pd.DataFrame({
    "recipe_ingredient": recipe_names,
    "carbon_entity": [
        carbon_lookup.get(x) for x in recipe_names
    ]
})

carbon_exact = carbon_exact.dropna(
    subset=["carbon_entity"]
).drop_duplicates("recipe_ingredient")

# ---------- FlavorDB ----------
def alias_values(value):
    if pd.isna(value):
        return []
    text = clean_name(value)
    if not text:
        return []
    parts = re.split(r"[;,|]", text)
    return [clean_name(x) for x in parts if clean_name(x)]

flavor_fields = [
    "entity_alias",
    "entity_alias_readable",
    "entity_alias_synonyms"
]

flavor_lookup = {}

for _, row in flavor_entities.iterrows():
    for field in flavor_fields:
        if field not in flavor_entities.columns:
            continue
        for alias in alias_values(row[field]):
            if alias and alias not in flavor_lookup:
                flavor_lookup[alias] = row["entity_alias"]

flavor_exact = pd.DataFrame({
    "recipe_ingredient": recipe_names,
    "flavor_entity": [
        flavor_lookup.get(x) for x in recipe_names
    ]
})

flavor_exact = flavor_exact.dropna(
    subset=["flavor_entity"]
).drop_duplicates("recipe_ingredient")

occ = (
    recipe_ingredients.groupby("ingredient_name")
    .size()
    .rename("occurrences")
)

carbon_exact_occ = (
    carbon_exact.merge(
        occ,
        left_on="recipe_ingredient",
        right_index=True,
        how="left"
    )["occurrences"]
    .sum()
)

flavor_exact_occ = (
    flavor_exact.merge(
        occ,
        left_on="recipe_ingredient",
        right_index=True,
        how="left"
    )["occurrences"]
    .sum()
)

total_types = len(recipe_names)
total_occ = len(recipe_ingredients)

baseline = pd.DataFrame({
    "dataset": ["SustainableFoodDB", "FlavorDB"],
    "exact_mapped_types": [
        len(carbon_exact),
        len(flavor_exact)
    ],
    "type_coverage_pct": [
        100 * len(carbon_exact) / total_types,
        100 * len(flavor_exact) / total_types
    ],
    "exact_mapped_occurrences": [
        carbon_exact_occ,
        flavor_exact_occ
    ],
    "occurrence_coverage_pct": [
        100 * carbon_exact_occ / total_occ,
        100 * flavor_exact_occ / total_occ
    ]
})

display(baseline.round(2))

carbon_exact.to_csv(
    RESULTS / "carbon_exact_mapping.csv", index=False
)

flavor_exact.to_csv(
    RESULTS / "flavor_exact_mapping.csv", index=False
)

baseline.to_csv(
    RESULTS / "mapping_baseline.csv", index=False
)


,dataset,exact_mapped_types,type_coverage_pct,exact_mapped_occurrences,occurrence_coverage_pct
0,SustainableFoodDB,2645,46.46,111782,69.03
1,FlavorDB,245,4.30,63073,38.95


## 3. Unmatched ingredients

This is retained as a compact diagnostic rather than the earlier substring-search cells. It shows where naming variation remains after exact matching.


In [35]:
# ================================================================
# CELL 3 — UNMATCHED INGREDIENT FREQUENCY
# ================================================================

carbon_exact_set = set(carbon_exact["recipe_ingredient"])
flavor_exact_set = set(flavor_exact["recipe_ingredient"])

freq = (
    recipe_ingredients.groupby("ingredient_name")
    .size()
    .sort_values(ascending=False)
)

carbon_unmatched = (
    freq[~freq.index.isin(carbon_exact_set)]
    .rename("occurrences")
    .reset_index()
    .rename(columns={"ingredient_name": "recipe_ingredient"})
)

flavor_unmatched = (
    freq[~freq.index.isin(flavor_exact_set)]
    .rename("occurrences")
    .reset_index()
    .rename(columns={"ingredient_name": "recipe_ingredient"})
)

print("Carbon unmatched types      :", len(carbon_unmatched))
print("Carbon unmatched occurrences:", carbon_unmatched["occurrences"].sum())

print("\nFlavorDB unmatched types      :", len(flavor_unmatched))
print("FlavorDB unmatched occurrences:", flavor_unmatched["occurrences"].sum())

print("\nTop 30 Carbon unmatched:")
display(carbon_unmatched.head(30))

print("\nTop 30 FlavorDB unmatched:")
display(flavor_unmatched.head(30))

carbon_unmatched.to_csv(
    RESULTS / "carbon_unmatched_frequency.csv", index=False
)

flavor_unmatched.to_csv(
    RESULTS / "flavor_unmatched_frequency.csv", index=False
)


Carbon unmatched types      : 3048
Carbon unmatched occurrences: 50152

FlavorDB unmatched types      : 5448
FlavorDB unmatched occurrences: 98861

Top 30 Carbon unmatched:


,recipe_ingredient,occurrences
0,salt,6977
1,black pepper,2249
2,parsley,1896
3,oregano,1452
4,baking powder,1089
5,extra virgin olive oil,1029
6,thyme,967
7,brown sugar,965
8,basil,793
9,bay leaf,772



Top 30 FlavorDB unmatched:


,recipe_ingredient,occurrences
0,salt,6977
1,sugar,5003
2,olive oil,3961
3,water,3287
4,garlic clove,2770
5,flour,2487
6,black pepper,2249
7,lemon juice,1955
8,purpose flour,1819
9,salt pepper,1697


## 4. Controlled normalization

Basic whitespace/case normalization produced essentially no useful gain. Instead of broad substring matching—which produced noisy relationships—we retain only manually reviewed, high-confidence naming variants from the earlier analysis.

These mappings are **database-join normalizations**, not claims that the two ingredients are chemically, nutritionally, or functionally identical.


In [36]:
# ================================================================
# CELL 4 — CONTROLLED NORMALIZATION
# ================================================================

carbon_safe = {
    "active yeast": "yeast",
    "black olive": "olive",
    "black pepper": "pepper",
    "brown sugar": "sugar",
    "cayenne pepper": "pepper",
    "celery rib": "celery",
    "cold water": "water",
    "creme fraiche": "creme",
    "extra virgin olive oil": "olive",
    "french bread": "bread",
    "heavy cream": "cream",
    "maple syrup": "syrup",
    "pine nut": "nut",
    "white bread": "bread",
    "white pepper": "pepper",
    "white sugar": "sugar",
}

flavor_safe = {
    "beef broth": "beef",
    "black olive": "olive",
    "black pepper": "pepper",
    "brown sugar": "sugar",
    "caster sugar": "sugar",
    "cayenne pepper": "pepper",
    "chicken breast": "chicken",
    "chicken broth": "chicken",
    "chicken stock": "chicken",
    "confectioner ' sugar": "sugar",
    "dijon mustard": "mustard",
    "egg white": "egg",
    "egg yolk": "egg",
    "garlic powder": "garlic",
    "green onion": "onion",
    "heavy cream": "cream",
    "kalamata olive": "olive",
    "kosher salt": "salt",
    "lemon juice": "lemon",
    "lemon zest": "lemon",
    "olive oil": "olive",
    "orange juice": "orange",
    "plain flour": "flour",
    "purpose flour": "flour",
    "red bell pepper": "pepper",
    "red onion": "onion",
    "red pepper": "pepper",
    "sea salt": "salt",
    "tomato paste": "tomato",
    "vanilla extract": "vanilla",
    "whipping cream": "cream",
    "white sugar": "sugar",
}

def apply_controlled_mapping(exact_df, mapping, target_lookup):
    rows = []

    exact_names = set(exact_df["recipe_ingredient"])

    for recipe_name in recipe_names:
        if recipe_name in exact_names:
            target = exact_df.loc[
                exact_df["recipe_ingredient"] == recipe_name
            ].iloc[0, 1]
            rows.append(
                (recipe_name, target, "exact")
            )
            continue

        normalized = mapping.get(recipe_name)
        if normalized is None:
            continue

        target = target_lookup.get(normalized)
        if target is not None:
            rows.append(
                (recipe_name, target, "controlled_normalization")
            )

    return pd.DataFrame(
        rows,
        columns=[
            "recipe_ingredient",
            "mapped_entity",
            "mapping_type"
        ]
    )

carbon_entity_lookup = {}
for _, row in carbon.iterrows():
    key = clean_name(row["Ingredient"])
    if key and key not in carbon_entity_lookup:
        carbon_entity_lookup[key] = row["Ingredient"]

flavor_entity_lookup = flavor_lookup

carbon_mapping = apply_controlled_mapping(
    carbon_exact.rename(columns={"carbon_entity": "mapped_entity"}),
    carbon_safe,
    carbon_entity_lookup
)

flavor_mapping = apply_controlled_mapping(
    flavor_exact.rename(columns={"flavor_entity": "mapped_entity"}),
    flavor_safe,
    flavor_entity_lookup
)

carbon_mapping.to_csv(
    RESULTS / "carbon_recipe_ingredient_mapping.csv",
    index=False
)

flavor_mapping.to_csv(
    RESULTS / "flavor_recipe_ingredient_mapping.csv",
    index=False
)

print("Carbon mappings:", len(carbon_mapping))
print("FlavorDB mappings:", len(flavor_mapping))

print("\nCarbon controlled mappings:")
display(
    carbon_mapping[
        carbon_mapping["mapping_type"] == "controlled_normalization"
    ]
)

print("\nFlavorDB controlled mappings:")
display(
    flavor_mapping[
        flavor_mapping["mapping_type"] == "controlled_normalization"
    ]
)


Carbon mappings: 2661
FlavorDB mappings: 277

Carbon controlled mappings:


,recipe_ingredient,mapped_entity,mapping_type
6,active yeast,yeast,controlled_normalization
275,black olive,olive,controlled_normalization
276,black pepper,pepper,controlled_normalization
355,brown sugar,sugar,controlled_normalization
471,cayenne pepper,pepper,controlled_normalization
477,celery rib,celery,controlled_normalization
706,cold water,water,controlled_normalization
798,creme fraiche,creme,controlled_normalization
923,extra virgin olive oil,olive,controlled_normalization
1006,french bread,bread,controlled_normalization



FlavorDB controlled mappings:


,recipe_ingredient,mapped_entity,mapping_type
22,beef broth,beef,controlled_normalization
28,black olive,olive,controlled_normalization
29,black pepper,pepper,controlled_normalization
38,brown sugar,NaN,controlled_normalization
51,caster sugar,NaN,controlled_normalization
54,cayenne pepper,pepper,controlled_normalization
65,chicken breast,chicken,controlled_normalization
66,chicken broth,chicken,controlled_normalization
67,chicken stock,chicken,controlled_normalization
79,confectioner ' sugar,NaN,controlled_normalization


## 5. Mapping coverage after normalization

In [37]:
# ================================================================
# CELL 5 — COVERAGE AFTER CONTROLLED NORMALIZATION
# ================================================================

def coverage_row(name, mapping):
    mapped_types = set(mapping["recipe_ingredient"])

    mapped_occ = (
        freq[freq.index.isin(mapped_types)].sum()
    )

    return {
        "dataset": name,
        "mapped_types": len(mapped_types),
        "type_coverage_pct": 100 * len(mapped_types) / total_types,
        "mapped_occurrences": mapped_occ,
        "occurrence_coverage_pct": 100 * mapped_occ / total_occ,
    }

coverage = pd.DataFrame([
    coverage_row("SustainableFoodDB", carbon_mapping),
    coverage_row("FlavorDB", flavor_mapping)
])

display(coverage.round(2))

coverage.to_csv(
    RESULTS / "mapping_coverage_after_normalization.csv",
    index=False
)


,dataset,mapped_types,type_coverage_pct,mapped_occurrences,occurrence_coverage_pct
0,SustainableFoodDB,2661,46.74,120184,74.22
1,FlavorDB,277,4.87,87176,53.83


In [38]:
# ================================================================
# CELL 6 — NORMALIZATION CONTRIBUTION
# ================================================================

def normalization_contribution(mapping):
    norm = mapping[
        mapping["mapping_type"] == "controlled_normalization"
    ].copy()

    if norm.empty:
        return norm

    norm["occurrences"] = norm["recipe_ingredient"].map(freq)

    return norm.sort_values(
        "occurrences",
        ascending=False
    )

carbon_contribution = normalization_contribution(
    carbon_mapping
)

flavor_contribution = normalization_contribution(
    flavor_mapping
)

print("Carbon normalized mappings:",
      len(carbon_contribution))
print("Recovered Carbon occurrences:",
      carbon_contribution["occurrences"].sum())

display(carbon_contribution)

print("\nFlavorDB normalized mappings:",
      len(flavor_contribution))
print("Recovered FlavorDB occurrences:",
      flavor_contribution["occurrences"].sum())

display(flavor_contribution)

carbon_contribution.to_csv(
    RESULTS / "carbon_normalization_contribution.csv",
    index=False
)

flavor_contribution.to_csv(
    RESULTS / "flavor_normalization_contribution.csv",
    index=False
)


Carbon normalized mappings: 16
Recovered Carbon occurrences: 8402


,recipe_ingredient,mapped_entity,mapping_type,occurrences
276,black pepper,pepper,controlled_normalization,2249
923,extra virgin olive oil,olive,controlled_normalization,1029
355,brown sugar,sugar,controlled_normalization,965
1238,heavy cream,cream,controlled_normalization,763
2618,white sugar,sugar,controlled_normalization,650
275,black olive,olive,controlled_normalization,357
471,cayenne pepper,pepper,controlled_normalization,323
2616,white pepper,pepper,controlled_normalization,320
1006,french bread,bread,controlled_normalization,293
1488,maple syrup,syrup,controlled_normalization,266



FlavorDB normalized mappings: 32
Recovered FlavorDB occurrences: 24103


,recipe_ingredient,mapped_entity,mapping_type,occurrences
170,olive oil,olive,controlled_normalization,3961
29,black pepper,pepper,controlled_normalization,2249
139,lemon juice,lemon,controlled_normalization,1955
203,purpose flour,NaN,controlled_normalization,1819
264,vanilla extract,vanilla,controlled_normalization,1131
38,brown sugar,NaN,controlled_normalization,965
97,egg yolk,egg,controlled_normalization,904
125,heavy cream,NaN,controlled_normalization,763
212,red onion,onion,controlled_normalization,719
273,white sugar,NaN,controlled_normalization,650


In [39]:
# ================================================================
# CELL 7 — COVERAGE CONCENTRATION BY INGREDIENT FREQUENCY
# ================================================================

frequency_table = (
    freq.rename("occurrences")
    .reset_index()
    .rename(columns={"ingredient_name": "ingredient"})
)

frequency_table["cumulative_occurrences"] = (
    frequency_table["occurrences"].cumsum()
)

frequency_table["cumulative_pct"] = (
    100 * frequency_table["cumulative_occurrences"]
    / total_occ
)

thresholds = [50, 75, 80, 90, 95, 99]

threshold_rows = []
for target in thresholds:
    hit = frequency_table[
        frequency_table["cumulative_pct"] >= target
    ].iloc[0]

    threshold_rows.append({
        "coverage_target_pct": target,
        "ingredient_types_required": int(hit.name + 1),
    })

threshold_table = pd.DataFrame(threshold_rows)

display(threshold_table)

top_k = [10, 25, 50, 100, 250, 500, 1000, 2500, 5000]

concentration_rows = []

carbon_set = set(carbon_mapping["recipe_ingredient"])
flavor_set = set(flavor_mapping["recipe_ingredient"])

for k in top_k:
    top_names = set(frequency_table.head(k)["ingredient"])

    concentration_rows.append({
        "top_k": k,
        "occurrences": frequency_table.head(k)["occurrences"].sum(),
        "carbon_coverage_pct": 100 * freq[
            freq.index.isin(top_names & carbon_set)
        ].sum() / total_occ,
        "flavor_coverage_pct": 100 * freq[
            freq.index.isin(top_names & flavor_set)
        ].sum() / total_occ,
    })

concentration = pd.DataFrame(concentration_rows)

display(concentration.round(2))

threshold_table.to_csv(
    RESULTS / "frequency_coverage_thresholds.csv",
    index=False
)

concentration.to_csv(
    RESULTS / "mapping_coverage_concentration.csv",
    index=False
)


,coverage_target_pct,ingredient_types_required
0,50,39
1,75,160
2,80,225
3,90,551
4,95,1147
5,99,4074


,top_k,occurrences,carbon_coverage_pct,flavor_coverage_pct
0,10,43707,22.68,14.32
1,25,68370,35.84,27.65
2,50,87928,44.46,36.75
3,100,108429,54.20,46.57
4,250,131925,64.38,51.84
5,500,144360,68.76,53.23
6,1000,152645,71.49,53.67
7,2500,158561,73.34,53.82
8,5000,161241,74.03,53.83


## 6. FlavorDB molecular representation

We now move from ingredient-name mapping to the molecular representation used for candidate generation.


In [40]:
# ================================================================
# CELL 8 — FLAVORDB MOLECULAR COVERAGE
# ================================================================

# Resolve mapped RecipeDB ingredients to FlavorDB entity IDs.
entity_name_to_ids = {}

for _, row in flavor_entities.iterrows():
    entity_name = clean_name(row["entity_alias"])
    if not entity_name:
        continue
    entity_name_to_ids.setdefault(
        entity_name, []
    ).append(row["entity_id"])

recipe_flavor_entity_rows = []

for _, row in flavor_mapping.iterrows():
    entity_name = clean_name(row["mapped_entity"])
    ids = entity_name_to_ids.get(entity_name, [])

    for entity_id in ids:
        recipe_flavor_entity_rows.append({
            "recipe_ingredient": row["recipe_ingredient"],
            "flavor_entity": row["mapped_entity"],
            "entity_id": entity_id,
            "mapping_type": row["mapping_type"]
        })

recipe_flavor_entity_mapping = pd.DataFrame(
    recipe_flavor_entity_rows
)

recipe_flavor_entity_mapping.to_csv(
    RESULTS / "recipe_flavor_entity_mapping.csv",
    index=False
)

recipe_flavor_molecules = (
    recipe_flavor_entity_mapping
    .merge(
        flavor_links[["entity_id", "pubchem_id"]].drop_duplicates(),
        on="entity_id",
        how="left"
    )
)

recipe_flavor_molecules = recipe_flavor_molecules.dropna(
    subset=["pubchem_id"]
)

recipe_flavor_molecules["pubchem_id"] = (
    recipe_flavor_molecules["pubchem_id"]
    .astype(str)
)

molecular_data = (
    recipe_flavor_molecules
    .merge(
        flavor_molecules,
        left_on="pubchem_id",
        right_on=flavor_molecules["pubchem_id"].astype(str),
        how="left",
        suffixes=("", "_molecule")
    )
)

ingredient_molecule_counts = (
    recipe_flavor_molecules
    .groupby("recipe_ingredient")["pubchem_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("Recipe ingredient mappings:",
      len(flavor_mapping))

print("Mapped ingredients with molecules:",
      ingredient_molecule_counts.size)

print("Unique PubChem molecules:",
      recipe_flavor_molecules["pubchem_id"].nunique())

print("\nMolecule-count statistics:")
display(
    ingredient_molecule_counts.describe().round(2)
)

print("\nTop 30 ingredients by recorded molecule count:")
display(
    ingredient_molecule_counts.head(30)
)

recipe_flavor_molecules.to_csv(
    RESULTS / "recipe_flavor_molecules.csv",
    index=False
)

molecular_data.to_csv(
    RESULTS / "recipe_flavor_molecular_data.csv",
    index=False
)


Recipe ingredient mappings: 277
Mapped ingredients with molecules: 267
Unique PubChem molecules: 1546

Molecule-count statistics:


count    267.00
mean     127.92
std       71.19
min        1.00
25%       86.00
50%      131.00
75%      172.00
max      391.00
Name: pubchem_id, dtype: float64


Top 30 ingredients by recorded molecule count:


recipe_ingredient
tea                391
apple              300
cocoa              294
tomato paste       271
tomato             271
papaya             270
coffee             269
orange juice       268
orange             268
seville orange     268
beer               267
grape              261
strawberry         257
potato             254
ginger             251
mango              238
spearmint          237
mushroom           227
peppermint         227
soybean            225
white wine         225
black pepper       223
red pepper         223
pepper             223
red bell pepper    223
cayenne pepper     223
pineapple          220
lemon zest         218
lemon              218
capsicum           218
Name: pubchem_id, dtype: int64

In [41]:
# ================================================================
# CELL 9 — MOLECULAR REPRESENTATION SANITY CHECK
# ================================================================

ingredient_molecules = (
    recipe_flavor_molecules
    .groupby("recipe_ingredient")["pubchem_id"]
    .apply(lambda x: set(x))
    .to_dict()
)

set_groups = {}

for ingredient, mols in ingredient_molecules.items():
    key = frozenset(mols)
    set_groups.setdefault(key, []).append(ingredient)

identical_sets = [
    {
        "ingredients": sorted(ingredients),
        "molecule_count": len(mols)
    }
    for mols, ingredients in set_groups.items()
    if len(ingredients) > 1
]

identical_sets = sorted(
    identical_sets,
    key=lambda x: (-x["molecule_count"], x["ingredients"])
)

print("Ingredients with molecular data:",
      len(ingredient_molecules))

print("Identical molecular sets shared by >=2 ingredients:",
      len(identical_sets))

display(pd.DataFrame(identical_sets).head(30))

pd.DataFrame(identical_sets).to_csv(
    RESULTS / "identical_flavor_molecular_sets.csv",
    index=False
)

print("""
Interpretation:
Identical molecular sets are useful evidence that the FlavorDB representation
contains entity variants/aliases, but they must NOT automatically be interpreted
as interchangeable culinary ingredients.
""")


Ingredients with molecular data: 267
Identical molecular sets shared by >=2 ingredients: 29


,ingredients,molecule_count
0,"[tomato, tomato paste]",271
1,"[orange, orange juice, seville orange]",268
2,"[black pepper, cayenne pepper, pepper, red bel...",223
3,"[lemon, lemon juice, lemon zest]",218
4,"[peanut, peanut butter]",215
5,"[nectarine, peach]",199
6,"[green onion, onion, red onion]",180
7,"[marjoram, oregano]",180
8,"[butter bean, lima bean]",174
9,"[vanilla, vanilla extract]",174



Interpretation:
Identical molecular sets are useful evidence that the FlavorDB representation
contains entity variants/aliases, but they must NOT automatically be interpreted
as interchangeable culinary ingredients.



## 7. Molecular similarity baseline

The earlier raw-vs-weighted experiments are condensed here into one reproducible baseline.

We use an IDF-weighted Jaccard-style overlap:
- ubiquitous molecules receive lower weight;
- rarer shared molecules receive higher weight.

This is an exploratory molecular signal. It is **not** a validated sensory-flavor metric, and rare single-molecule overlaps can still produce misleading scores.


In [42]:
# ================================================================
# CELL 10 — RARITY-WEIGHTED MOLECULAR SIMILARITY
# ================================================================

ingredient_names = sorted(ingredient_molecules)

molecule_ingredient_counts = Counter()

for mols in ingredient_molecules.values():
    for mol in mols:
        molecule_ingredient_counts[mol] += 1

N = len(ingredient_molecules)

molecule_idf = {
    mol: math.log(N / count)
    for mol, count in molecule_ingredient_counts.items()
}

pair_rows = []

for a, b in combinations(ingredient_names, 2):
    mols_a = ingredient_molecules[a]
    mols_b = ingredient_molecules[b]

    shared = mols_a & mols_b

    if not shared:
        continue

    union = mols_a | mols_b

    raw_jaccard = len(shared) / len(union)

    numerator = sum(
        molecule_idf[m]
        for m in shared
    )

    denominator = sum(
        molecule_idf[m]
        for m in union
    )

    weighted_jaccard = (
        numerator / denominator
        if denominator > 0 else 0.0
    )

    pair_rows.append({
        "ingredient_a": a,
        "ingredient_b": b,
        "molecules_a": len(mols_a),
        "molecules_b": len(mols_b),
        "shared_molecules": len(shared),
        "raw_jaccard": raw_jaccard,
        "molecular_similarity": weighted_jaccard,
        "size_ratio": min(len(mols_a), len(mols_b))
                       / max(len(mols_a), len(mols_b)),
    })

molecular_similarity = pd.DataFrame(pair_rows)

print("Ingredients with molecular data:",
      len(ingredient_molecules))

print("Unique molecules:",
      len(molecule_idf))

print("Ingredient pairs with >=1 shared molecule:",
      len(molecular_similarity))

print("Mean molecular similarity:",
      round(molecular_similarity["molecular_similarity"].mean(), 4))

print("Median molecular similarity:",
      round(molecular_similarity["molecular_similarity"].median(), 4))

print("Maximum molecular similarity:",
      round(molecular_similarity["molecular_similarity"].max(), 4))

print("\nTop molecular relationships:")
display(
    molecular_similarity
    .sort_values(
        ["molecular_similarity", "shared_molecules"],
        ascending=False
    )
    .head(30)
)

molecular_similarity.to_csv(
    RESULTS / "final_molecular_similarity.csv",
    index=False
)

pd.DataFrame([
    {
        "pubchem_id": mol,
        "ingredient_count": count,
        "idf_weight": molecule_idf[mol]
    }
    for mol, count in molecule_ingredient_counts.items()
]).to_csv(
    RESULTS / "molecule_rarity_weights.csv",
    index=False
)


Ingredients with molecular data: 267
Unique molecules: 1546
Ingredient pairs with >=1 shared molecule: 33513
Mean molecular similarity: 0.105
Median molecular similarity: 0.0668
Maximum molecular similarity: 1.0

Top molecular relationships:


,ingredient_a,ingredient_b,molecules_a,molecules_b,shared_molecules,raw_jaccard,molecular_similarity,size_ratio
33365,tomato,tomato paste,271,271,271,1.0,1.0,1.0
28832,orange,orange juice,268,268,268,1.0,1.0,1.0
28889,orange,seville orange,268,268,268,1.0,1.0,1.0
28986,orange juice,seville orange,268,268,268,1.0,1.0,1.0
7036,black pepper,cayenne pepper,223,223,223,1.0,1.0,1.0
7161,black pepper,pepper,223,223,223,1.0,1.0,1.0
7181,black pepper,red bell pepper,223,223,223,1.0,1.0,1.0
7185,black pepper,red pepper,223,223,223,1.0,1.0,1.0
12137,cayenne pepper,pepper,223,223,223,1.0,1.0,1.0
12157,cayenne pepper,red bell pepper,223,223,223,1.0,1.0,1.0


In [43]:
# ================================================================
# CELL 11 — MOLECULAR SIMILARITY LIMITATIONS / REFERENCE CHECK
# ================================================================

reference_pairs = [
    ("beef", "beef broth"),
    ("chicken", "chicken broth"),
    ("chicken", "chicken breast"),
    ("tomato", "tomato paste"),
    ("vanilla", "vanilla extract"),
    ("olive", "olive oil"),
    ("onion", "red onion"),
    ("onion", "green onion"),
    ("garlic", "garlic powder"),
    ("lemon", "lemon juice"),
    ("orange", "orange juice"),
    ("egg", "egg white"),
    ("beef", "pork"),
    ("chicken", "pork"),
    ("lobster", "prawn"),
    ("onion", "garlic"),
    ("black pepper", "nutmeg"),
    ("black pepper", "rosemary"),
]

lookup = {}

for _, row in molecular_similarity.iterrows():
    lookup[
        (row["ingredient_a"], row["ingredient_b"])
    ] = row

    lookup[
        (row["ingredient_b"], row["ingredient_a"])
    ] = row

reference_rows = []

for a, b in reference_pairs:
    row = lookup.get((a, b))

    if row is None:
        continue

    reference_rows.append({
        "ingredient_a": a,
        "ingredient_b": b,
        "shared_molecules": row["shared_molecules"],
        "raw_jaccard": row["raw_jaccard"],
        "molecular_similarity": row["molecular_similarity"],
        "size_ratio": row["size_ratio"],
    })

reference_table = pd.DataFrame(reference_rows)

display(reference_table)

reference_table.to_csv(
    RESULTS / "molecular_similarity_reference_pairs.csv",
    index=False
)

print("""
Key methodological warning:
molecular overlap is a useful candidate-generation signal, but it is not by
itself a reliable substitution decision. Identical database molecular sets,
molecule-set-size imbalance, and rare shared molecules can all create
misleadingly high similarity.
""")


,ingredient_a,ingredient_b,shared_molecules,raw_jaccard,molecular_similarity,size_ratio
0,beef,beef broth,92,1.000000,1.000000,1.000000
1,chicken,chicken broth,131,1.000000,1.000000,1.000000
2,chicken,chicken breast,131,1.000000,1.000000,1.000000
3,tomato,tomato paste,271,1.000000,1.000000,1.000000
4,vanilla,vanilla extract,174,1.000000,1.000000,1.000000
5,olive,olive oil,155,1.000000,1.000000,1.000000
6,onion,red onion,180,1.000000,1.000000,1.000000
7,onion,green onion,180,1.000000,1.000000,1.000000
8,garlic,garlic powder,143,1.000000,1.000000,1.000000
9,lemon,lemon juice,218,1.000000,1.000000,1.000000



Key methodological warning:
molecular overlap is a useful candidate-generation signal, but it is not by
itself a reliable substitution decision. Identical database molecular sets,
molecule-set-size imbalance, and rare shared molecules can all create
misleadingly high similarity.



## 8. Candidate generation

Every molecularly connected ingredient pair becomes a preliminary candidate. Carbon information is attached here, but carbon reduction is not yet used as the sole selection criterion.


In [44]:
# ================================================================
# CELL 12 — INITIAL CANDIDATE PAIRS + CARBON
# ================================================================

# Carbon footprint lookup.
carbon_values = (
    carbon.assign(
        carbon_key=carbon["Ingredient"].map(clean_name)
    )
    .dropna(subset=["carbon_key"])
    .groupby("carbon_key")["CarbonFootprint"]
    .first()
)

candidate_pairs = molecular_similarity.copy()

candidate_pairs["carbon_a"] = (
    candidate_pairs["ingredient_a"]
    .map(carbon_values)
)

candidate_pairs["carbon_b"] = (
    candidate_pairs["ingredient_b"]
    .map(carbon_values)
)

candidate_pairs["has_carbon_both"] = (
    candidate_pairs["carbon_a"].notna()
    & candidate_pairs["carbon_b"].notna()
)

candidate_pairs["carbon_reduction"] = np.where(
    candidate_pairs["has_carbon_both"],
    (
        candidate_pairs["carbon_a"]
        - candidate_pairs["carbon_b"]
    ) / candidate_pairs["carbon_a"].replace(0, np.nan),
    np.nan
)

candidate_pairs["carbon_delta"] = np.where(
    candidate_pairs["has_carbon_both"],
    candidate_pairs["carbon_b"]
    - candidate_pairs["carbon_a"],
    np.nan
)

candidate_pairs = candidate_pairs.sort_values(
    "molecular_similarity",
    ascending=False
).reset_index(drop=True)

print("Candidate ingredient pairs       :", len(candidate_pairs))
print("Pairs with carbon for both       :",
      candidate_pairs["has_carbon_both"].sum())
print("Pairs missing carbon information :",
      (~candidate_pairs["has_carbon_both"]).sum())

display(
    candidate_pairs.head(30)
)

candidate_pairs.to_csv(
    RESULTS / "initial_candidate_pairs.csv",
    index=False
)


Candidate ingredient pairs       : 33513
Pairs with carbon for both       : 15947
Pairs missing carbon information : 17566


,ingredient_a,ingredient_b,molecules_a,molecules_b,shared_molecules,raw_jaccard,molecular_similarity,size_ratio,carbon_a,carbon_b,has_carbon_both,carbon_reduction,carbon_delta
0,tomato,tomato paste,271,271,271,1.0,1.0,1.0,0.48,1.28,True,-1.666667,0.80
1,vanilla,vanilla extract,174,174,174,1.0,1.0,1.0,4.30,4.30,True,0.000000,0.00
2,brandy,cognac,50,50,50,1.0,1.0,1.0,NaN,NaN,False,NaN,NaN
3,black olive,kalamata olive,155,155,155,1.0,1.0,1.0,NaN,0.59,False,NaN,NaN
4,red bell pepper,red pepper,223,223,223,1.0,1.0,1.0,0.80,0.80,True,0.000000,0.00
5,kalamata olive,olive,155,155,155,1.0,1.0,1.0,0.59,0.59,True,0.000000,0.00
6,kalamata olive,olive oil,155,155,155,1.0,1.0,1.0,0.59,3.84,True,-5.508475,3.25
7,marjoram,oregano,180,180,180,1.0,1.0,1.0,1.43,NaN,False,NaN,NaN
8,red kidney bean,string bean,91,91,91,1.0,1.0,1.0,NaN,0.67,False,NaN,NaN
9,dijon mustard,mustard oil,135,135,135,1.0,1.0,1.0,NaN,NaN,False,NaN,NaN


## 9. RecipeDB co-occurrence compatibility

Co-occurrence is used as a **compatibility/context signal**, not as proof of substitution. A pair that already occurs together frequently is more naturally interpreted as a complement; absence of co-occurrence is only a preliminary guard.


In [45]:
# ================================================================
# CELL 13 — RECIPEDB CO-OCCURRENCE
# ================================================================

recipe_sets = (
    recipe_ingredients
    .groupby("recipe_id")["ingredient_name"]
    .apply(lambda x: sorted(set(x)))
)

pair_counts = Counter()
ingredient_recipe_counts = Counter()

for ingredients in recipe_sets:
    for ingredient in ingredients:
        ingredient_recipe_counts[ingredient] += 1

    for a, b in combinations(ingredients, 2):
        pair_counts[(a, b)] += 1

cooccurrence_rows = []

for (a, b), count in pair_counts.items():
    fa = ingredient_recipe_counts[a]
    fb = ingredient_recipe_counts[b]

    union = fa + fb - count
    jaccard = count / union if union else 0.0

    cooccurrence_rows.append({
        "ingredient_a": a,
        "ingredient_b": b,
        "cooccurrence_count": count,
        "occurrences_a": fa,
        "occurrences_b": fb,
        "cooccurrence_jaccard": jaccard,
        "p_b_given_a": count / fa if fa else 0.0,
        "p_a_given_b": count / fb if fb else 0.0,
    })

cooccurrence = pd.DataFrame(cooccurrence_rows)

candidate_pairs_with_compatibility = (
    candidate_pairs
    .merge(
        cooccurrence,
        on=["ingredient_a", "ingredient_b"],
        how="left"
    )
)

# Pairs absent from RecipeDB co-occurrence receive zero.
cooccurrence_cols = [
    "cooccurrence_count",
    "occurrences_a",
    "occurrences_b",
    "cooccurrence_jaccard",
    "p_b_given_a",
    "p_a_given_b",
]

for col in cooccurrence_cols:
    candidate_pairs_with_compatibility[col] = (
        candidate_pairs_with_compatibility[col]
        .fillna(0)
    )

print("Recipes:", len(recipe_sets))
print("Unique ingredients:", len(ingredient_recipe_counts))
print("Unique ingredient pairs with co-occurrence:",
      len(cooccurrence))

candidate_pairs_with_compatibility.to_csv(
    RESULTS / "candidate_pairs_with_compatibility.csv",
    index=False
)

display(
    candidate_pairs_with_compatibility
    .sort_values(
        ["cooccurrence_count", "cooccurrence_jaccard"],
        ascending=False
    )
    .head(30)
)


Recipes: 16971
Unique ingredients: 5693
Unique ingredient pairs with co-occurrence: 157329


,ingredient_a,ingredient_b,molecules_a,molecules_b,shared_molecules,raw_jaccard,molecular_similarity,size_ratio,carbon_a,carbon_b,has_carbon_both,carbon_reduction,carbon_delta,cooccurrence_count,occurrences_a,occurrences_b,cooccurrence_jaccard,p_b_given_a,p_a_given_b
13479,butter,egg,131,55,28,0.177215,0.101667,0.419847,9.90,3.23,True,0.673737,-6.67,2639.0,6359.0,4659.0,0.314954,0.415002,0.566431
8253,egg,milk,55,111,33,0.248120,0.154613,0.495495,3.23,5.34,True,-0.653251,2.11,1909.0,4659.0,3178.0,0.322031,0.409745,0.600692
2597,butter,milk,131,111,64,0.359551,0.252144,0.847328,9.90,5.34,True,0.460606,-4.56,1869.0,6359.0,3178.0,0.243740,0.293914,0.588106
20694,butter,onion,131,180,31,0.110714,0.045403,0.727778,9.90,0.24,True,0.975758,-9.66,1259.0,6359.0,3727.0,0.142631,0.197987,0.337805
9327,olive oil,onion,155,180,99,0.419492,0.141428,0.861111,3.84,0.24,True,0.937500,-3.60,1235.0,3685.0,3727.0,0.199935,0.335142,0.331366
26711,cinnamon,egg,183,55,17,0.076923,0.024385,0.300546,4.30,3.23,True,0.248837,-1.07,940.0,1805.0,4659.0,0.170167,0.520776,0.201760
20752,butter,cinnamon,131,183,30,0.105634,0.045206,0.715847,9.90,4.30,True,0.565657,-5.60,933.0,6359.0,1805.0,0.129028,0.146721,0.516898
6713,olive oil,tomato,155,271,127,0.424749,0.172606,0.571956,3.84,0.48,True,0.875000,-3.36,914.0,3685.0,1796.0,0.200131,0.248033,0.508909
10517,black pepper,olive oil,223,155,103,0.374545,0.128977,0.695067,NaN,3.84,False,NaN,NaN,841.0,2177.0,3685.0,0.167497,0.386311,0.228223
9688,onion,tomato,180,271,118,0.354354,0.137734,0.664207,0.24,0.48,True,-1.000000,0.24,828.0,3727.0,1796.0,0.176358,0.222163,0.461024


## 10. Sustainability-aware screening

This is the final output of `EDA_05_mapping`.

The screening rule is deliberately conservative:

1. molecular similarity ≥ 0.10
2. carbon footprint known for both ingredients
3. candidate reduces carbon footprint
4. candidate does not already directly co-occur in RecipeDB

The **0.10 molecular threshold is a screening threshold, not a scientifically validated substitution cutoff**. GastroGraph will later add category, embedding, and graph-based signals before final substitution decisions are made.


In [46]:
# ================================================================
# CELL 14 — FINAL SUSTAINABILITY SCREENING
# ================================================================

screened_candidates = candidate_pairs_with_compatibility.copy()

screened_candidates = screened_candidates[
    screened_candidates["molecular_similarity"] >= 0.10
].copy()

screened_candidates = screened_candidates[
    screened_candidates["has_carbon_both"]
].copy()

screened_candidates = screened_candidates[
    screened_candidates["carbon_reduction"] > 0
].copy()

screened_candidates = screened_candidates[
    screened_candidates["cooccurrence_count"] == 0
].copy()

screened_candidates["molecular_signal"] = True
screened_candidates["carbon_signal"] = True
screened_candidates["no_recipe_cooccurrence"] = True

screened_candidates = screened_candidates.sort_values(
    [
        "molecular_similarity",
        "carbon_reduction",
        "shared_molecules"
    ],
    ascending=[False, False, False]
).reset_index(drop=True)

print("=" * 70)
print("FINAL SCREENING")
print("=" * 70)

original = candidate_pairs_with_compatibility

stage_1 = original[
    original["molecular_similarity"] >= 0.10
]

stage_2 = stage_1[
    stage_1["has_carbon_both"]
]

stage_3 = stage_2[
    stage_2["carbon_reduction"] > 0
]

print(f"Original molecular candidate pairs : {len(original):,}")
print(f"Molecular similarity >= 0.10       : {len(stage_1):,}")
print(f"With carbon data for both           : {len(stage_2):,}")
print(f"Carbon-reducing candidates          : {len(stage_3):,}")
print(f"Potential substitution candidates   : {len(screened_candidates):,}")

display_columns = [
    "ingredient_a",
    "ingredient_b",
    "molecules_a",
    "molecules_b",
    "shared_molecules",
    "molecular_similarity",
    "size_ratio",
    "occurrences_a",
    "occurrences_b",
    "carbon_a",
    "carbon_b",
    "carbon_reduction",
    "cooccurrence_count",
    "cooccurrence_jaccard",
    "p_b_given_a",
    "p_a_given_b",
]

display(
    screened_candidates[
        [c for c in display_columns
         if c in screened_candidates.columns]
    ].head(50)
)

output_path = RESULTS / "sustainability_screened_candidates.csv"

screened_candidates.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)
print("\nEDA_05_MAPPING COMPLETE")


FINAL SCREENING
Original molecular candidate pairs : 33,513
Molecular similarity >= 0.10       : 13,661
With carbon data for both           : 6,915
Carbon-reducing candidates          : 3,537
Potential substitution candidates   : 2,193


,ingredient_a,ingredient_b,molecules_a,molecules_b,shared_molecules,molecular_similarity,size_ratio,occurrences_a,occurrences_b,carbon_a,carbon_b,carbon_reduction,cooccurrence_count,cooccurrence_jaccard,p_b_given_a,p_a_given_b
0,aubergine,eggplant,132,132,132,1.000000,1.000000,0.0,0.0,7.55,0.45,0.940397,0.0,0.0,0.0,0.0
1,pimento,pimiento,12,12,12,1.000000,1.000000,0.0,0.0,6.13,0.96,0.843393,0.0,0.0,0.0,0.0
2,haricot bean,string bean,91,91,91,1.000000,1.000000,0.0,0.0,1.32,0.67,0.492424,0.0,0.0,0.0,0.0
3,flageolet bean,string bean,91,91,91,1.000000,1.000000,0.0,0.0,1.32,0.67,0.492424,0.0,0.0,0.0,0.0
4,flageolet bean,green bean,91,91,91,1.000000,1.000000,0.0,0.0,1.32,0.73,0.446970,0.0,0.0,0.0,0.0
5,orange juice,seville orange,268,268,268,1.000000,1.000000,0.0,0.0,0.62,0.35,0.435484,0.0,0.0,0.0,0.0
6,green bean,string bean,91,91,91,1.000000,1.000000,0.0,0.0,0.73,0.67,0.082192,0.0,0.0,0.0,0.0
7,cheese,goat cheese,117,117,113,0.901822,1.000000,0.0,0.0,11.87,8.18,0.310868,0.0,0.0,0.0,0.0
8,cottage cheese,goat cheese,117,117,113,0.871561,1.000000,0.0,0.0,11.87,8.18,0.310868,0.0,0.0,0.0,0.0
9,limburger cheese,mozzarella cheese,116,120,113,0.866953,0.966667,0.0,0.0,11.87,8.65,0.271272,0.0,0.0,0.0,0.0



Saved: c:\Users\Parth\semester7\BTP_2\BTP_2\EDA\results\mapping\sustainability_screened_candidates.csv

EDA_05_MAPPING COMPLETE


# Handoff to GastroGraph

The notebook ends here deliberately.

The next stage should **not** add more ad-hoc similarity experiments. It should map RecipeDB ingredient names into GastroGraph and add its:

- GraphSAGE embedding cosine similarity
- TF-IDF-weighted molecular overlap
- culinary category / functional group
- direct graph co-occurrence
- product filtering / compatibility logic

GastroGraph's actual implementation will be used rather than reimplementing or guessing its API.

The resulting combined candidate table can then feed the BTP's multi-objective optimization layer.
